# Notebook_1_SQL_Silver_Load — Silver Layer
### O&G Rig Operations Intelligence Platform
**Layer:** Bronze → Silver  
**Run after:** Notebook_0c_Bronze_Raw_Load  
**Strategy:** Full rebuild every run — silver is always clean and derived from bronze  

| Table | Source |
|---|---|
| silver_rig_ops | fact_rig_daily_ops + 8 dims |
| silver_equipment_events | fact_equipment_events + 5 dims |
| silver_maintenance_log | fact_maintenance_log + 6 dims |
| silver_crew_hours | fact_crew_hours + 5 dims |
| silver_incident_log | fact_incident_log + 7 dims |

In [0]:
-- CELL 1 — SETUP
CREATE SCHEMA IF NOT EXISTS workspace.silver;

In [0]:
-- CELL 2 — silver_rig_ops
-- Joins fact_rig_daily_ops to 8 dims
-- Adds derived metrics, business flags, clean measures
-- All dim joins filter is_current = true (SCD Type 2)

CREATE OR REPLACE TABLE workspace.silver.silver_rig_ops
USING DELTA
AS
SELECT
    -- date attributes
    f.date_id,
    d.full_date                                         AS operation_date,
    d.year_actual,
    d.month_actual,
    d.month_name,
    d.month_name_abbreviated,
    d.quarter_actual,
    d.quarter_name,
    d.week_of_year,
    d.is_weekend,

    -- rig attributes
    f.rig_id,
    r.rig_name,
    r.rig_type,
    r.commission_year,

    -- region attributes
    rg.region_name,
    rg.basin,
    rg.state,

    -- well attributes
    w.well_name,
    wt.well_type,

    -- status attributes
    st.status_name                                      AS rig_status,
    st.status_category,

    -- shift attributes
    sh.shift_name,
    sh.start_hour,
    sh.end_hour,

    -- downtime reason attributes
    dr.downtime_reason,
    dr.downtime_category,
    dr.is_controllable                                  AS is_controllable_downtime,

    -- cost category attributes
    cc.cost_category,
    cc.cost_type,

    -- raw measures
    f.drilling_hours,
    f.downtime_hours,
    f.npt_hours,
    f.rop_ft_hr,
    f.cost_per_day,
    f.sla_met,

    -- derived metrics
    ROUND(f.drilling_hours / 24 * 100, 2)               AS uptime_pct,
    ROUND(f.downtime_hours / 24 * 100, 2)               AS downtime_pct,
    CASE
        WHEN f.drilling_hours + f.downtime_hours > 0
        THEN ROUND(f.npt_hours / (f.drilling_hours + f.downtime_hours) * 100, 2)
        ELSE 0
    END                                                 AS npt_pct,

    -- business flags
    CASE WHEN f.sla_met = 0        THEN 1 ELSE 0 END    AS is_sla_breach,
    CASE WHEN f.drilling_hours > 24 THEN 1 ELSE 0 END   AS is_invalid_hours,
    CASE WHEN f.cost_per_day < 0   THEN 1 ELSE 0 END    AS is_invalid_cost,
    CASE WHEN f.rop_ft_hr IS NULL  THEN 1 ELSE 0 END    AS is_missing_rop,

    -- data quality flag
    CASE
        WHEN f.drilling_hours > 24 OR f.cost_per_day < 0
        THEN 0 ELSE 1
    END                                                 AS is_valid,

    -- clean measures (silver corrections)
    CASE WHEN f.drilling_hours > 24 THEN 24.0
         ELSE f.drilling_hours END                      AS drilling_hours_clean,
    CASE WHEN f.cost_per_day < 0 THEN ABS(f.cost_per_day)
         ELSE f.cost_per_day END                        AS cost_per_day_clean,
    COALESCE(f.rop_ft_hr, 0.0)                         AS rop_ft_hr_clean,

    -- pipeline metadata
    f.loaded_at                                         AS bronze_loaded_at,
    current_timestamp()                                 AS transformed_at

FROM workspace.bronze.fact_rig_daily_ops f

LEFT JOIN workspace.bronze.dim_date d
    ON  f.date_id = d.date_id

LEFT JOIN workspace.bronze.dim_rig r
    ON  f.rig_id_sk    = r.rig_id_sk
    AND r.is_current   = true

LEFT JOIN workspace.bronze.dim_region rg
    ON  f.region_id_sk = rg.region_id_sk
    AND rg.is_current  = true

LEFT JOIN workspace.bronze.dim_well w
    ON  f.well_id_sk   = w.well_id_sk
    AND w.is_current   = true

LEFT JOIN workspace.bronze.dim_well_type wt
    ON  f.well_type_id_sk = wt.well_type_id_sk
    AND wt.is_current     = true

LEFT JOIN workspace.bronze.dim_status st
    ON  f.status_id_sk = st.status_id_sk
    AND st.is_current  = true

LEFT JOIN workspace.bronze.dim_shift sh
    ON  f.shift_id_sk  = sh.shift_id_sk
    AND sh.is_current  = true

LEFT JOIN workspace.bronze.dim_downtime_reason dr
    ON  f.downtime_reason_id_sk = dr.downtime_reason_id_sk
    AND dr.is_current           = true

LEFT JOIN workspace.bronze.dim_cost_category cc
    ON  f.cost_category_id_sk = cc.cost_category_id_sk
    AND cc.is_current         = true;

num_affected_rows,num_inserted_rows


In [0]:
-- Quality check — silver_rig_ops
SELECT
    COUNT(*)                                                        AS total_rows,
    SUM(CASE WHEN is_valid = 1          THEN 1 ELSE 0 END)         AS valid_rows,
    SUM(CASE WHEN is_valid = 0          THEN 1 ELSE 0 END)         AS invalid_rows,
    SUM(CASE WHEN drilling_hours IS NULL THEN 1 ELSE 0 END)        AS null_drilling_hours,
    SUM(CASE WHEN rop_ft_hr IS NULL     THEN 1 ELSE 0 END)        AS null_rop,
    SUM(CASE WHEN downtime_reason IS NULL THEN 1 ELSE 0 END)      AS null_downtime_reason,
    SUM(CASE WHEN rig_id IS NULL        THEN 1 ELSE 0 END)        AS null_rig_id
FROM workspace.silver.silver_rig_ops;

total_rows,valid_rows,invalid_rows,null_drilling_hours,null_rop,null_downtime_reason,null_rig_id
17620,17620,0,0,0,0,0


In [0]:
-- CELL 3 — silver_equipment_events
-- Joins fact_equipment_events to 5 dims
-- Adds SLA metrics, downtime flags, failure categorisation

CREATE OR REPLACE TABLE workspace.silver.silver_equipment_events
USING DELTA
AS
SELECT
    -- natural business ID
    f.event_id,

    -- date attributes
    f.date_id,
    d.full_date                                             AS event_date,
    d.year_actual,
    d.month_actual,
    d.month_name,
    d.quarter_actual,
    d.quarter_name,

    -- rig attributes
    f.rig_id,
    r.rig_name,
    r.rig_type,

    -- region attributes
    rg.region_name,
    rg.basin,

    -- equipment attributes
    e.equipment_type,
    e.criticality,
    e.expected_lifespan_days,

    -- failure attributes
    ft.failure_type,
    ft.failure_category,
    ft.severity                                             AS failure_severity,

    -- measures
    f.failure_flag,
    f.downtime_caused_hrs,
    f.sla_met,
    f.resolution_hrs,

    -- business flags
    CASE WHEN f.sla_met = 0 THEN 1 ELSE 0 END               AS is_sla_breach,
    ROUND(
        f.downtime_caused_hrs / NULLIF(f.resolution_hrs, 0) * 100, 2
    )                                                       AS downtime_to_resolution_pct,

    -- data quality flag
    CASE
        WHEN f.downtime_caused_hrs < 0 OR f.resolution_hrs < 0
        THEN 0 ELSE 1
    END                                                     AS is_valid,

    -- pipeline metadata
    f.loaded_at                                             AS bronze_loaded_at,
    current_timestamp()                                     AS transformed_at

FROM workspace.bronze.fact_equipment_events f

LEFT JOIN workspace.bronze.dim_date d
    ON  f.date_id = d.date_id

LEFT JOIN workspace.bronze.dim_rig r
    ON  f.rig_id_sk   = r.rig_id_sk
    AND r.is_current  = true

LEFT JOIN workspace.bronze.dim_region rg
    ON  f.region_id_sk = rg.region_id_sk
    AND rg.is_current  = true

LEFT JOIN workspace.bronze.dim_equipment e
    ON  f.equipment_id_sk = e.equipment_id_sk
    AND e.is_current      = true

LEFT JOIN workspace.bronze.dim_failure_type ft
    ON  f.failure_type_id_sk = ft.failure_type_id_sk
    AND ft.is_current        = true;

num_affected_rows,num_inserted_rows


In [0]:
-- Quality check — silver_equipment_events
SELECT
    COUNT(*)                                                        AS total_rows,
    SUM(CASE WHEN is_valid = 1             THEN 1 ELSE 0 END)      AS valid_rows,
    SUM(CASE WHEN is_valid = 0             THEN 1 ELSE 0 END)      AS invalid_rows,
    SUM(CASE WHEN equipment_type IS NULL   THEN 1 ELSE 0 END)      AS null_equipment_type,
    SUM(CASE WHEN failure_type IS NULL     THEN 1 ELSE 0 END)      AS null_failure_type,
    SUM(CASE WHEN downtime_caused_hrs IS NULL THEN 1 ELSE 0 END)   AS null_downtime_hrs,
    SUM(CASE WHEN rig_id IS NULL           THEN 1 ELSE 0 END)      AS null_rig_id
FROM workspace.silver.silver_equipment_events;

total_rows,valid_rows,invalid_rows,null_equipment_type,null_failure_type,null_downtime_hrs,null_rig_id
88100,88100,0,0,0,0,0


In [0]:
-- CELL 4 — silver_maintenance_log
-- Joins fact_maintenance_log to 6 dims
-- Adds cost efficiency metrics, planning flags

CREATE OR REPLACE TABLE workspace.silver.silver_maintenance_log
USING DELTA
AS
SELECT
    -- natural business ID
    f.maintenance_id,

    -- date attributes
    f.date_id,
    d.full_date                                             AS maintenance_date,
    d.year_actual,
    d.month_actual,
    d.month_name,
    d.quarter_actual,
    d.quarter_name,

    -- rig attributes
    f.rig_id,
    r.rig_name,
    r.rig_type,

    -- region attributes
    rg.region_name,
    rg.basin,

    -- equipment attributes
    e.equipment_type,
    e.criticality,

    -- maintenance type attributes
    mt.maintenance_type,
    mt.schedule_type,
    mt.is_planned,

    -- contractor attributes
    c.contractor_name,
    c.contractor_type,

    -- measures
    f.duration_hrs,
    f.cost,
    f.technician_count,

    -- derived metrics
    ROUND(f.cost / NULLIF(f.duration_hrs, 0), 2)           AS cost_per_hour,
    ROUND(f.cost / NULLIF(f.technician_count, 0), 2)       AS cost_per_technician,

    -- business flags
    CASE WHEN mt.is_planned = false THEN 0 ELSE 1 END       AS is_preventive,
    CASE WHEN f.duration_hrs > 24   THEN 1 ELSE 0 END       AS is_long_maintenance,
    CASE WHEN f.cost > 50000        THEN 1 ELSE 0 END       AS is_high_cost,

    -- data quality flag
    CASE
        WHEN f.cost < 0 OR f.duration_hrs < 0
        THEN 0 ELSE 1
    END                                                     AS is_valid,

    -- pipeline metadata
    f.loaded_at                                             AS bronze_loaded_at,
    current_timestamp()                                     AS transformed_at

FROM workspace.bronze.fact_maintenance_log f

LEFT JOIN workspace.bronze.dim_date d
    ON  f.date_id = d.date_id

LEFT JOIN workspace.bronze.dim_rig r
    ON  f.rig_id_sk   = r.rig_id_sk
    AND r.is_current  = true

LEFT JOIN workspace.bronze.dim_region rg
    ON  f.region_id_sk = rg.region_id_sk
    AND rg.is_current  = true

LEFT JOIN workspace.bronze.dim_equipment e
    ON  f.equipment_id_sk = e.equipment_id_sk
    AND e.is_current      = true

LEFT JOIN workspace.bronze.dim_maintenance_type mt
    ON  f.maintenance_type_id_sk = mt.maintenance_type_id_sk
    AND mt.is_current            = true

LEFT JOIN workspace.bronze.dim_contractor c
    ON  f.contractor_id_sk = c.contractor_id_sk
    AND c.is_current       = true;

num_affected_rows,num_inserted_rows


In [0]:
-- Quality check — silver_maintenance_log
SELECT
    COUNT(*)                                                        AS total_rows,
    SUM(CASE WHEN is_valid = 1             THEN 1 ELSE 0 END)      AS valid_rows,
    SUM(CASE WHEN is_valid = 0             THEN 1 ELSE 0 END)      AS invalid_rows,
    SUM(CASE WHEN equipment_type IS NULL   THEN 1 ELSE 0 END)      AS null_equipment_type,
    SUM(CASE WHEN cost IS NULL             THEN 1 ELSE 0 END)      AS null_cost,
    SUM(CASE WHEN duration_hrs IS NULL     THEN 1 ELSE 0 END)      AS null_duration,
    SUM(CASE WHEN rig_id IS NULL           THEN 1 ELSE 0 END)      AS null_rig_id
FROM workspace.silver.silver_maintenance_log;

total_rows,valid_rows,invalid_rows,null_equipment_type,null_cost,null_duration,null_rig_id
8672,8672,0,0,0,0,0


In [0]:
-- CELL 5 — silver_crew_hours
-- Joins fact_crew_hours to 5 dims
-- Adds shift analysis flags, NPT attribution

CREATE OR REPLACE TABLE workspace.silver.silver_crew_hours
USING DELTA
AS
SELECT
    -- date attributes
    f.date_id,
    d.full_date                                             AS work_date,
    d.year_actual,
    d.month_actual,
    d.month_name,
    d.quarter_actual,
    d.quarter_name,
    d.week_of_year,

    -- rig attributes
    f.rig_id,
    r.rig_name,
    r.rig_type,

    -- region attributes
    rg.region_name,
    rg.basin,

    -- crew attributes
    f.crew_id,
    c.role,
    c.certification_level,
    c.years_experience,

    -- shift attributes
    sh.shift_name,
    sh.start_hour,
    sh.end_hour,

    -- measures
    f.hours_worked,
    f.overtime_hours,
    f.is_present,
    f.npt_attributed,

    -- derived metrics
    f.hours_worked + f.overtime_hours                       AS total_hours,

    -- business flags
    CASE WHEN f.overtime_hours > 0  THEN 1 ELSE 0 END       AS has_overtime,
    CASE WHEN f.hours_worked < 8    THEN 1 ELSE 0 END       AS is_short_shift,
    CASE WHEN f.npt_attributed = 1  THEN 1 ELSE 0 END       AS is_npt_crew,

    -- data quality flag
    CASE
        WHEN f.hours_worked < 0 OR f.hours_worked > 24
        THEN 0 ELSE 1
    END                                                     AS is_valid,

    -- pipeline metadata
    f.loaded_at                                             AS bronze_loaded_at,
    current_timestamp()                                     AS transformed_at

FROM workspace.bronze.fact_crew_hours f

LEFT JOIN workspace.bronze.dim_date d
    ON  f.date_id = d.date_id

LEFT JOIN workspace.bronze.dim_rig r
    ON  f.rig_id_sk   = r.rig_id_sk
    AND r.is_current  = true

LEFT JOIN workspace.bronze.dim_region rg
    ON  f.region_id_sk = rg.region_id_sk
    AND rg.is_current  = true

LEFT JOIN workspace.bronze.dim_crew_member c
    ON  f.crew_id_sk  = c.crew_id_sk
    AND c.is_current  = true

LEFT JOIN workspace.bronze.dim_shift sh
    ON  f.shift_id_sk = sh.shift_id_sk
    AND sh.is_current = true;

num_affected_rows,num_inserted_rows


In [0]:
-- Quality check — silver_crew_hours
SELECT
    COUNT(*)                                                        AS total_rows,
    SUM(CASE WHEN is_valid = 1             THEN 1 ELSE 0 END)      AS valid_rows,
    SUM(CASE WHEN is_valid = 0             THEN 1 ELSE 0 END)      AS invalid_rows,
    SUM(CASE WHEN crew_id IS NULL          THEN 1 ELSE 0 END)      AS null_crew_id,
    SUM(CASE WHEN hours_worked IS NULL     THEN 1 ELSE 0 END)      AS null_hours_worked,
    SUM(CASE WHEN shift_name IS NULL       THEN 1 ELSE 0 END)      AS null_shift,
    SUM(CASE WHEN rig_id IS NULL           THEN 1 ELSE 0 END)      AS null_rig_id
FROM workspace.silver.silver_crew_hours;

total_rows,valid_rows,invalid_rows,null_crew_id,null_hours_worked,null_shift,null_rig_id
35240,35240,0,0,0,0,0


In [0]:
-- CELL 6 — silver_incident_log
-- Joins fact_incident_log to 7 dims
-- Adds severity ranking, downtime causation flags

CREATE OR REPLACE TABLE workspace.silver.silver_incident_log
USING DELTA
AS
SELECT
    -- natural business ID
    f.incident_id,

    -- date attributes
    f.date_id,
    d.full_date                                             AS incident_date,
    d.year_actual,
    d.month_actual,
    d.month_name,
    d.quarter_actual,
    d.quarter_name,

    -- rig attributes
    f.rig_id,
    r.rig_name,
    r.rig_type,

    -- region attributes
    rg.region_name,
    rg.basin,

    -- crew attributes
    c.crew_id,
    c.role                                                  AS crew_role,
    c.certification_level,
    c.years_experience,

    -- incident type attributes
    it.incident_type,
    it.incident_category,
    it.severity_level,

    -- equipment attributes
    e.equipment_type,
    e.criticality                                           AS equipment_criticality,

    -- status attributes
    st.status_name                                          AS incident_status,
    st.status_category,

    -- measures
    f.downtime_caused_hrs,
    f.is_recordable,

    -- business flags
    CASE WHEN f.is_recordable = 1          THEN 1 ELSE 0 END  AS is_serious_incident,
    CASE WHEN f.downtime_caused_hrs > 0    THEN 1 ELSE 0 END  AS caused_downtime,
    CASE
        WHEN it.severity_level = 'High'   THEN 1
        WHEN it.severity_level = 'Medium' THEN 2
        ELSE 3
    END                                                     AS severity_rank,

    -- data quality flag
    CASE
        WHEN f.downtime_caused_hrs < 0
        THEN 0 ELSE 1
    END                                                     AS is_valid,

    -- pipeline metadata
    f.loaded_at                                             AS bronze_loaded_at,
    current_timestamp()                                     AS transformed_at

FROM workspace.bronze.fact_incident_log f

LEFT JOIN workspace.bronze.dim_date d
    ON  f.date_id = d.date_id

LEFT JOIN workspace.bronze.dim_rig r
    ON  f.rig_id_sk    = r.rig_id_sk
    AND r.is_current   = true

LEFT JOIN workspace.bronze.dim_region rg
    ON  f.region_id_sk = rg.region_id_sk
    AND rg.is_current  = true

LEFT JOIN workspace.bronze.dim_crew_member c
    ON  f.crew_id_sk   = c.crew_id_sk
    AND c.is_current   = true

LEFT JOIN workspace.bronze.dim_incident_type it
    ON  f.incident_type_id_sk = it.incident_type_id_sk
    AND it.is_current         = true

LEFT JOIN workspace.bronze.dim_equipment e
    ON  f.equipment_id_sk = e.equipment_id_sk
    AND e.is_current      = true

LEFT JOIN workspace.bronze.dim_status st
    ON  f.status_id_sk = st.status_id_sk
    AND st.is_current  = true;

num_affected_rows,num_inserted_rows


In [0]:
-- Quality check — silver_incident_log
SELECT
    COUNT(*)                                                        AS total_rows,
    SUM(CASE WHEN is_valid = 1             THEN 1 ELSE 0 END)      AS valid_rows,
    SUM(CASE WHEN is_valid = 0             THEN 1 ELSE 0 END)      AS invalid_rows,
    SUM(CASE WHEN equipment_type IS NULL   THEN 1 ELSE 0 END)      AS null_equipment_type,
    SUM(CASE WHEN severity_level IS NULL   THEN 1 ELSE 0 END)      AS null_severity,
    SUM(CASE WHEN downtime_caused_hrs IS NULL THEN 1 ELSE 0 END)   AS null_downtime_hrs,
    SUM(CASE WHEN rig_id IS NULL           THEN 1 ELSE 0 END)      AS null_rig_id
FROM workspace.silver.silver_incident_log;

total_rows,valid_rows,invalid_rows,null_equipment_type,null_severity,null_downtime_hrs,null_rig_id
1465,1465,0,0,0,0,0


In [0]:
-- CELL 7 — FINAL VALIDATION
-- Row counts, transformed_at check, is_valid check across all silver tables

SELECT
    table_name,
    row_count,
    valid_rows,
    invalid_rows,
    ROUND(valid_rows / row_count * 100, 1)  AS valid_pct,
    has_transformed_at,
    has_is_valid
FROM (
    SELECT
        'silver_rig_ops'            AS table_name,
        COUNT(*)                    AS row_count,
        SUM(CASE WHEN is_valid = 1  THEN 1 ELSE 0 END) AS valid_rows,
        SUM(CASE WHEN is_valid = 0  THEN 1 ELSE 0 END) AS invalid_rows,
        MAX(CASE WHEN transformed_at IS NOT NULL THEN 1 ELSE 0 END) AS has_transformed_at,
        MAX(CASE WHEN is_valid IS NOT NULL       THEN 1 ELSE 0 END) AS has_is_valid
    FROM workspace.silver.silver_rig_ops

    UNION ALL
    SELECT
        'silver_equipment_events',
        COUNT(*),
        SUM(CASE WHEN is_valid = 1 THEN 1 ELSE 0 END),
        SUM(CASE WHEN is_valid = 0 THEN 1 ELSE 0 END),
        MAX(CASE WHEN transformed_at IS NOT NULL THEN 1 ELSE 0 END),
        MAX(CASE WHEN is_valid IS NOT NULL       THEN 1 ELSE 0 END)
    FROM workspace.silver.silver_equipment_events

    UNION ALL
    SELECT
        'silver_maintenance_log',
        COUNT(*),
        SUM(CASE WHEN is_valid = 1 THEN 1 ELSE 0 END),
        SUM(CASE WHEN is_valid = 0 THEN 1 ELSE 0 END),
        MAX(CASE WHEN transformed_at IS NOT NULL THEN 1 ELSE 0 END),
        MAX(CASE WHEN is_valid IS NOT NULL       THEN 1 ELSE 0 END)
    FROM workspace.silver.silver_maintenance_log

    UNION ALL
    SELECT
        'silver_crew_hours',
        COUNT(*),
        SUM(CASE WHEN is_valid = 1 THEN 1 ELSE 0 END),
        SUM(CASE WHEN is_valid = 0 THEN 1 ELSE 0 END),
        MAX(CASE WHEN transformed_at IS NOT NULL THEN 1 ELSE 0 END),
        MAX(CASE WHEN is_valid IS NOT NULL       THEN 1 ELSE 0 END)
    FROM workspace.silver.silver_crew_hours

    UNION ALL
    SELECT
        'silver_incident_log',
        COUNT(*),
        SUM(CASE WHEN is_valid = 1 THEN 1 ELSE 0 END),
        SUM(CASE WHEN is_valid = 0 THEN 1 ELSE 0 END),
        MAX(CASE WHEN transformed_at IS NOT NULL THEN 1 ELSE 0 END),
        MAX(CASE WHEN is_valid IS NOT NULL       THEN 1 ELSE 0 END)
    FROM workspace.silver.silver_incident_log
)
ORDER BY table_name;

table_name,row_count,valid_rows,invalid_rows,valid_pct,has_transformed_at,has_is_valid
silver_crew_hours,35240,35240,0,100.0,1,1
silver_equipment_events,88100,88100,0,100.0,1,1
silver_incident_log,1465,1465,0,100.0,1,1
silver_maintenance_log,8672,8672,0,100.0,1,1
silver_rig_ops,17620,17620,0,100.0,1,1
